# AERIS Exposure: Dynamic Exposure Assessment

This notebook demonstrates a complete dynamic-mode workflow:
1. Load GPS trajectory data
2. Run the Google Maps Air Quality estimator
3. Interpret multi-pollutant exposure summaries
4. Export to GeoJSON for GIS integration

In [ ]:
import pandas as pd
from aeris.exposure import ExposureEstimatorGoogleMapsDynamic

## 1. Prepare Input Data

Dynamic mode requires:
- `subject_id` — participant identifier
- `timestamp` — ISO 8601 datetime with timezone
- `latitude`, `longitude` — GPS coordinates

Multiple subjects and multi-day trajectories are supported.

In [ ]:
df_input = pd.read_csv("dynamic_example.csv")
print(f"Subjects: {df_input['subject_id'].nunique()}")
print(f"GPS points: {len(df_input)}")
df_input.head()

## 2. Run the Estimator

The Google Maps backend returns all six pollutants (PM2.5, PM10, SO2, NO2, CO, O3)
for each GPS point.

In [ ]:
estimator = ExposureEstimatorGoogleMapsDynamic(
    input_csv="dynamic_example.csv",
    # gmaps_api_key="YOUR_KEY",  # or set GOOGLE_MAPS_AIR_QUALITY_API_KEY env var
)

result = estimator.run(skip_confirm=True)
result

## 3. Multi-Pollutant Summary

Each row is one subject-pollutant pair with:
- Concentration statistics (mean, std, median, min, max)
- EPA AQI value and category
- Time-weighted mean (accounts for temporal gaps)

In [ ]:
if not result.empty:
    # Pivot to see all pollutants side by side
    pivot = result.pivot_table(
        index="subject_id",
        columns="pollutant",
        values="mean",
    )
    pivot

## 4. GeoJSON Export

For GIS integration (QGIS, ArcGIS, kepler.gl), export as GeoJSON.
Requires `geopandas`: `pip install 'aeris[geo]'`

In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Point

    # Use raw output for per-point GeoJSON
    # For summary-level, use result DataFrame with station coordinates
    if not result.empty and "latitude" in df_input.columns:
        print("GeoJSON export available with per-point raw data.")
        print("Use --format geojson from CLI for automatic export.")
except ImportError:
    print("Install geopandas for GeoJSON support: pip install 'aeris[geo]'")